1. HMM

In [3]:
import numpy as np
import math


class HMM:


    def __init__(self, pi, A, B):
        self.pi     = np.array(pi, dtype=float)
        self.A      = np.array(A,  dtype=float)
        self.B      = np.array(B,  dtype=float)
        self.N      = len(pi)
        self.log_pi = np.log(self.pi + 1e-300)
        self.log_A  = np.log(self.A  + 1e-300)
        self.log_B  = np.log(self.B  + 1e-300)

    def _lse(self, x):
        m = np.max(x)
        return m + np.log(np.sum(np.exp(x - m))) if not np.isinf(m) else m

    def forward(self, seq):
        T  = len(seq)
        la = np.zeros((T, self.N))
        la[0] = self.log_pi + self.log_B[:, seq[0]]
        for t in range(1, T):
            for s in range(self.N):
                la[t, s] = self._lse(la[t-1] + self.log_A[:, s]) + self.log_B[s, seq[t]]
        return self._lse(la[-1]), la

    def backward(self, seq):
        T  = len(seq)
        lb = np.zeros((T, self.N))  # beta[-1] = log(1) = 0
        for t in range(T-2, -1, -1):
            for s in range(self.N):
                lb[t, s] = self._lse(self.log_A[s] + self.log_B[:, seq[t+1]] + lb[t+1])
        return lb

    def forward_backward(self, seq):
        log_p, la = self.forward(seq)
        lb        = self.backward(seq)
        return np.exp(la + lb - log_p)

    def viterbi(self, seq):
        T   = len(seq)
        ld  = np.zeros((T, self.N))
        psi = np.zeros((T, self.N), dtype=int)
        ld[0] = self.log_pi + self.log_B[:, seq[0]]
        for t in range(1, T):
            for s in range(self.N):
                cands     = ld[t-1] + self.log_A[:, s]
                psi[t, s] = np.argmax(cands)
                ld[t, s]  = cands[psi[t, s]] + self.log_B[s, seq[t]]
        path = [int(np.argmax(ld[-1]))]
        for t in range(T-1, 0, -1):
            path.append(psi[t, path[-1]])
        return path[::-1], float(np.max(ld[-1]))


2. Unit Test

In [4]:
pi = [0.5, 0.5]
A  = [[0.7, 0.3], [0.4, 0.6]]
B  = [[0.5, 0.5], [0.8, 0.2]]
hmm = HMM(pi, A, B)
seq = [0, 0, 0, 1, 0]  # H H H T H

log_p, la = hmm.forward(seq)
lb        = hmm.backward(seq)
gamma     = hmm.forward_backward(seq)
path, _   = hmm.viterbi(seq)
names     = ['Fair', 'Biased']

print(f'log P(seq)                    = {log_p:.6f}')
print(f'Forward == Forward-Backward   : {np.isclose(log_p, hmm._lse(la[0] + lb[0]))}')
print(f'Gamma sums to 1               : {np.allclose(gamma.sum(axis=1), 1.0)}')
print(f'Viterbi path: {" -> ".join(names[s] for s in path)}')

log P(seq)                    = -2.828027
Forward == Forward-Backward   : True
Gamma sums to 1               : True
Viterbi path: Biased -> Biased -> Biased -> Fair -> Fair


## FATHMM Score Computation

FATHMM uses pre-trained Pfam profile HMMs (learned via Baum-Welch on millions of sequences). We do **not** retrain the HMM — we use the pre-trained emission probabilities directly.

### How it works

1. **Viterbi** aligns the query protein sequence to a Pfam domain → maps each protein position to a domain position
2. Read **emission probabilities** from that domain position: P(amino acid | position)
3. Compute log-likelihood ratio: `log P(wt) / P(mut)`
4. Multiply by pathogenicity weight W_d (provided by FATHMM web server)

score = W_d × log( P(wt | domain_pos) / P(mut | domain_pos) )


score ≤ -1.5 → Damaging (Pathogenic)


score >  -1.5 → Tolerated (Benign)

### Connection to our HMM implementation

| Component | Our implementation | FATHMM |
|---|---|---|
| Learning (Baum-Welch) | Demonstrated in HMM class | Done by Pfam on millions of sequences |
| Alignment (Viterbi) | Implemented from scratch | Run via HMMER |
| Scoring | log P(wt) / P(mut) | Same, weighted by W_d |

The unweighted score we compute directly from Pfam emission probabilities should correlate with the weighted score from the FATHMM web server. The weight W_d amplifies the signal for domains that are known to harbour disease-causing mutations.